In [1]:
%pip install kaggle


   -------------------------- ------------- 2/3 [kaggle]
   -------------------------- ------------- 2/3 [kaggle]
   ---------------------------------------- 3/3 [kaggle]

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from pathlib import Path

# 1. Définir les identifiants AVANT d'importer la bibliothèque kaggle
os.environ['KAGGLE_USERNAME'] = "thomasbarascud"  # Remplacez par votre vrai username
os.environ['KAGGLE_KEY'] = "56fc415b99ab474ae4873d4880624d6f"        # Remplacez par votre vraie clé

# 2. Importer l'API et s'authentifier
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

# 3. Préparer le dossier
dataset_dir = Path("raw_dataset")
dataset_dir.mkdir(exist_ok=True)

print("Connexion réussie ! Téléchargement et extraction de CelebA-HQ (environ 3 Go)...")
print("Veuillez patienter, cela peut prendre plusieurs minutes sans afficher de barre de progression.")

# 4. Télécharger et décompresser via le code Python directement
api.dataset_download_files('lamsimon/celebahq', path=str(dataset_dir), unzip=True)

print("Téléchargement et extraction terminés avec succès !")

Connexion réussie ! Téléchargement et extraction de CelebA-HQ (environ 3 Go)...
Veuillez patienter, cela peut prendre plusieurs minutes sans afficher de barre de progression.
Dataset URL: https://www.kaggle.com/datasets/lamsimon/celebahq
Téléchargement et extraction terminés avec succès !


In [8]:
# Vérifier combien d'images ont été téléchargées
# CelebA-HQ contient généralement les images dans un sous-dossier (ex: celeba_hq_256 ou data)
image_files = list(dataset_dir.rglob("*.jpg")) + list(dataset_dir.rglob("*.png"))

print(f"Nombre d'images trouvées : {len(image_files)}")
if len(image_files) > 0:
    print(f"Exemple de chemin : {image_files[0]}")

Nombre d'images trouvées : 30000
Exemple de chemin : raw_dataset\celeba_hq\train\female\000004.jpg


In [9]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# On importe les outils de votre projet
from src.utils.config import Settings
from src.backends.insightface_backend import InsightFaceBackend
from src.utils.image_utils import load_image

# Initialisation d'InsightFace (en mode CPU)
settings = Settings(device="cpu")
# On désactive temporairement les logs trop verbeux
import logging
logging.getLogger().setLevel(logging.ERROR)

print("Chargement d'InsightFace...")
recognizer = InsightFaceBackend(settings)
print("Modèle chargé !")

Chargement d'InsightFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Modèle chargé !


In [13]:
import sys
# Ajout du dossier vendor pour accéder à l'outil d'alignement de SimSwap
sys.path.insert(0, str(settings.simswap))
from insightface_func.utils.face_align_ffhqandnewarc import estimate_norm

def align_and_crop(image_bgr, face, size=512):
    """Prend une image et un visage détecté, retourne le visage aligné en 512x512."""
    # "ffhq" est le standard d'alignement pour les modèles génératifs de visages
    matrix, _ = estimate_norm(face.landmarks, size, mode="ffhq")
    crop = cv2.warpAffine(image_bgr, matrix, (size, size), borderValue=0.0)
    return crop

In [ ]:
input_dir = Path("raw_dataset")
output_dir = Path("processed_dataset_512")
output_dir.mkdir(exist_ok=True)

processed_images = []

print("Début du traitement...")
for img_path in input_dir.rglob("*.*"):
    if img_path.suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
        continue
        
    try:
        # 1. Charger l'image
        image = load_image(img_path)
        
        # 2. Détecter les visages
        faces = recognizer.detect(image)
        
        if not faces:
            print(f"Aucun visage trouvé dans {img_path.name}")
            continue
            
        # 3. Prendre le visage principal (le plus grand)
        # On trie par la surface de la bounding box
        main_face = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]), reverse=True)[0]
        
        # 4. Aligner et recadrer en 512x512
        cropped_face = align_and_crop(image, main_face, size=512)
        
        # 5. Sauvegarder le résultat
        out_path = output_dir / f"{img_path.stem}_aligned.png"
        if out_path.exists():
            continue
        cv2.imwrite(str(out_path), cropped_face)
        
        # On garde en mémoire pour la visualisation
        processed_images.append(cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB))
        
    except Exception as e:
        print(f"Erreur sur {img_path.name} : {e}")

print(f"Terminé ! {len(processed_images)} visages traités.")

In [ ]:
# Afficher les 16 premières images générées
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    if i < len(processed_images):
        ax.imshow(processed_images[i])
    ax.axis('off')
plt.tight_layout()
plt.show()